# CRUD - Manipulation de la base NOSQL
1) Import des libraries
2) Connexion à la base à partir de localhost
3) Enregistrement des variables pour manipuler les collections
4) CREATE - Insertion d'un nouveau document dans la collection patient
5) READ - Lecture du document nouvellement insérée
6) UPDATE - Modification d'une variable dans un document
7) DELETE - Suppresion de la fiche patient nouvellement crée
8) LOOKUP - Jointure de la collection "encounters"(consultation) avec la table "patients"(fiche patient)
9) Verification de la jointure avec deux READ des collections
10) Affiche la jointure les deux collections ensemble 
11) EXPORT Export des 2 collections jointes en csv

### 1) Import des librairies orientées MongoDB

In [4]:
import os
import pandas as pd 
from dotenv import load_dotenv 
from pymongo import MongoClient 
from bson import ObjectId

### 2) Connexion au service mongodb dans le container en passant par le port 27017

In [6]:
load_dotenv()
  
client = MongoClient(
    host = 'localhost',
    port = 27017,
    username = 'root',
    password = 'example'
)

### 3) Création des variables attachées aux collections et à la base dans le service dans MongoDB  

In [7]:
db = client['medical_p5']
patients = db['patients']
facilities = db['healthcare_facility']
encounters = db['encounters']

### 4) CREATE - Insertion d'un nouveau document dans la collection patient

In [4]:
patients.insert_one({'Name': 'Son Goku', 'Age': 500, 'Gender': 'Male', 'Blood Type': 'X+', 'Medical Condition': 'Atrophia', 'Insurance Provider': 'Self'})

InsertOneResult(ObjectId('6a05ef97fcc3d5fcc7a84e55'), acknowledged=True)

### 5) READ - Lecture du document nouvellement insérée

In [5]:
print(patients.find_one({'Name':'Son Goku'}))

{'_id': ObjectId('6a05ef97fcc3d5fcc7a84e55'), 'Name': 'Son Goku', 'Age': 500, 'Gender': 'Male', 'Blood Type': 'X+', 'Medical Condition': 'Atrophia', 'Insurance Provider': 'Self'}


6) UPDATE - Modification d'une variable dans un document

In [6]:
patients.update_one({'Name':'Son Goku'},{'$set':{'Age': 501}})
print(patients.find_one({'Name':'Son Goku'}))

{'_id': ObjectId('6a05ef97fcc3d5fcc7a84e55'), 'Name': 'Son Goku', 'Age': 501, 'Gender': 'Male', 'Blood Type': 'X+', 'Medical Condition': 'Atrophia', 'Insurance Provider': 'Self'}


### 7) DELETE - Suppresion de la fiche patient nouvellement crée

In [7]:
patients.delete_one({'Name': 'Son Goku'})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

### 8) LOOKUP - Jointure de la collection "encounters"(consultation) avec la table "patients"(fiche patient)

In [8]:
jointure_enc_pat = db['encounters'].aggregate([
    {'$lookup':{
        'from':'patients', 
        'localField':'patient_id',
        'foreignField':'patient_id', 
        'as':'first_joint'
    }}
])

### 9) Verification de la jointure avec deux READ des collections

In [9]:
print(patients.find_one())

{'_id': ObjectId('6a0494e68a91ea0f9411e284'), 'patient_id': ObjectId('6a0494da8a91ea0f9410700a'), 'Name': 'Bobby JacksOn', 'Age': 30, 'Gender': 'Male', 'Blood Type': 'B-', 'Medical Condition': 'Cancer', 'Insurance Provider': 'Blue Cross'}


In [ ]:
print(encounters.find_one())
# Les numéros de patient_id correspondent

{'_id': ObjectId('6a0494e88a91ea0f941354fe'), 'patient_id': ObjectId('6a0494da8a91ea0f9410700a'), 'facility_id': ObjectId('6a0494e08a91ea0f941146c0'), 'Date of Admission': datetime.datetime(2024, 1, 31, 0, 0), 'Admission Type': 'Urgent', 'Room Number': 328, 'Doctor': 'Matthew Smith', 'Test Results': 'Normal', 'Medication': 'Paracetamol', 'Billing Amount': 18856.281305978155, 'Discharge Date': datetime.datetime(2024, 2, 2, 0, 0)}


### 10) Affiche la jointure les deux collections ensemble 

In [11]:
for doc in jointure_enc_pat:
    print(doc)
    break

{'_id': ObjectId('6a0494e88a91ea0f941354fe'), 'patient_id': ObjectId('6a0494da8a91ea0f9410700a'), 'facility_id': ObjectId('6a0494e08a91ea0f941146c0'), 'Date of Admission': datetime.datetime(2024, 1, 31, 0, 0), 'Admission Type': 'Urgent', 'Room Number': 328, 'Doctor': 'Matthew Smith', 'Test Results': 'Normal', 'Medication': 'Paracetamol', 'Billing Amount': 18856.281305978155, 'Discharge Date': datetime.datetime(2024, 2, 2, 0, 0), 'first_joint': [{'_id': ObjectId('6a0494e68a91ea0f9411e284'), 'patient_id': ObjectId('6a0494da8a91ea0f9410700a'), 'Name': 'Bobby JacksOn', 'Age': 30, 'Gender': 'Male', 'Blood Type': 'B-', 'Medical Condition': 'Cancer', 'Insurance Provider': 'Blue Cross'}]}


### 11) EXPORT - Export des 2 collections jointes en csv (à cause de la perte du curseur tout est refait)

In [14]:
# Jointure
jointure_enc_pat = db['encounters'].aggregate([
    {'$lookup':{
        'from':'patients', 
        'localField':'patient_id',
        'foreignField':'patient_id', 
        'as':'first_joint'
    }},
    {'$limit': 100}
])

# Export
df_export = pd.DataFrame(list(jointure_enc_pat))
df_export.to_csv('export_enc_pat.csv', index=False)

# Verification
print(df_export.shape)

(100, 12)


### 12) Export de chacune des collections en csv

In [17]:
# Export de la collection "patients"

df_export_patients = pd.DataFrame(list(patients.find())) # <le find ici est important il permet de garder les colonnes ce qui est plus interessant pour les tableurs et donc les utilisateurs
df_export_patients.to_csv('collection_patients.csv', index = False)
# Export de la collection "healthcare_facility"
df_export_facility = pd.DataFrame(list(facilities.find()))
df_export_facility.to_csv('collection_facility.csv', index = False)
# Export de la collection "encounters"
df_export_encounters = pd.DataFrame(list(encounters.find()))
df_export_encounters.to_csv('collection_encounters.csv', index = False)

# Affichage des résultats
print(df_export_patients.head(1))
print(df_export_facility.head(1))
print(df_export_encounters.head(1))


                        _id                patient_id           Name  Age  \
0  6a0494e68a91ea0f9411e284  6a0494da8a91ea0f9410700a  Bobby JacksOn   30   

  Gender Blood Type Medical Condition Insurance Provider  
0   Male         B-            Cancer         Blue Cross  
                        _id               facility_id         Hospital
0  6a0494e78a91ea0f9412b93a  6a0494e08a91ea0f941146c0  Sons and Miller
                        _id                patient_id  \
0  6a0494e88a91ea0f941354fe  6a0494da8a91ea0f9410700a   

                facility_id Date of Admission Admission Type  Room Number  \
0  6a0494e08a91ea0f941146c0        2024-01-31         Urgent          328   

          Doctor Test Results   Medication  Billing Amount Discharge Date  
0  Matthew Smith       Normal  Paracetamol    18856.281306     2024-02-02  


In [ ]:
# Bonus = L'idée c'est d'avoir des colonnes propres pour la table jointe et pas une liste csv collée

# Aplatir la colonne patient_info
df_patients = pd.json_normalize(df_export['first_joint'].apply(lambda x: x[0] if x else {}))

# Joindre les deux
df_final = pd.concat([df_export.drop('first_joint', axis=1), df_patients], axis=1)
df_final.to_csv('export_enc_pat.csv', index=False)
print(df_final.head())

                        _id                patient_id  \
0  6a0494e88a91ea0f941354fe  6a0494da8a91ea0f9410700a   
1  6a0494e88a91ea0f941354ff  6a0494da8a91ea0f9410700b   
2  6a0494e88a91ea0f94135500  6a0494da8a91ea0f9410700c   
3  6a0494e88a91ea0f94135501  6a0494da8a91ea0f9410700d   
4  6a0494e88a91ea0f94135502  6a0494da8a91ea0f9410700e   

                facility_id Date of Admission Admission Type  Room Number  \
0  6a0494e08a91ea0f941146c0        2024-01-31         Urgent          328   
1  6a0494e08a91ea0f941146c1        2019-08-20      Emergency          265   
2  6a0494e08a91ea0f941146c2        2022-09-22      Emergency          205   
3  6a0494e08a91ea0f941146c3        2020-11-18       Elective          450   
4  6a0494e08a91ea0f941146c4        2022-09-19         Urgent          458   

             Doctor  Test Results   Medication  Billing Amount Discharge Date  \
0     Matthew Smith        Normal  Paracetamol    18856.281306     2024-02-02   
1   Samantha Davies  Inconclusiv